---
title: "Data Collection"
format:
    html: 
        code-fold: false
---

{{< include instructions.qmd >}} 

{{< include overview.qmd >}} 

{{< include methods.qmd >}} 

# Code

Data are retrieved from the [Open Science Framework (OSF)](https://osf.io) through its public REST API (v2). Two OSF projects are used:

| OSF node | Role | Notes |
|---|---|---|
| [5mnuq](https://osf.io/5mnuq) | Primary dataset | Age-neutrality sample (Facon et al., 2025); retains respondents who failed the Conscientious Responder Scale |
| [hc9ur](https://osf.io/hc9ur) | Benchmark | Normative sample (Facon et al., 2026); already filtered on the CRS |

<!-- TODO (Christabel): explain in your own words why the primary file is the only one that can answer the research question -->

Participant-level data are sensitive and are never displayed on this site; the cells below output file metadata only.

## Setup

To set up for data collection, the following code block is run to import libraries and define constants. It points to the OSF API and the specific nodes where data is collected from, as well as the directory on my local machine where raw data is stored. The directory is created if it does not already exist.

In [1]:
from pathlib import Path
import hashlib

import pandas as pd
import pyreadstat
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

osf_api = "https://api.osf.io/v2"
raw_dir = Path("../../data/raw-data")
raw_dir.mkdir(parents=True, exist_ok=True)

# OSF projects ("nodes") used in this project
mynodes = {
    "5mnuq": "primary",
    "hc9ur": "benchmark",
}

# OSF's file server occasionally errors; added retry with backoff factor
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])))

## List files through the OSF API

Each node's `osfstorage` endpoint returns file metadata in a JSON, including a download link and the SHA-256 hash that OSF computed at upload.

In [5]:
def list_osf_files(node_id):
    """Return metadata for the top-level files in an OSF node's storage."""
    resp = session.get(f"{osf_api}/nodes/{node_id}/files/osfstorage/", timeout=30)
    resp.raise_for_status()
    rows = []
    for item in resp.json()["data"]:
        attrs = item["attributes"]
        if attrs["kind"] != "file":
            continue
        rows.append({
            "node": node_id,
            "role": mynodes[node_id],
            "name": attrs["name"],
            "sha256": attrs["extra"]["hashes"]["sha256"],
        })
    return rows


files = pd.DataFrame([row for node in mynodes for row in list_osf_files(node)])
files[["node", "role", "name", "sha256"]]

,node,role,name,sha256
0,5mnuq,primary,PhD_MF_Age-neutrality_Archived.sav,2caf1add2c6bc6de898e0423d3eb1a93f7f3f6f8472508...
1,hc9ur,benchmark,OSF_LPFSBF20_PID5BF+M_Community.sav,58b079f4e6a1fa1373f76c29b543d37a68f8c626a8065e...


## Download and verify

Raw data are kept pristine: an existing local file is accepted only if its hash matches OSF, and a mismatch raises an error instead of overwriting it.

In [6]:
def sha256sum(path):
    """SHA-256 hex digest of a file, read in chunks."""
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 16), b""):
            digest.update(chunk)
    return digest.hexdigest()

def download_verified(row, dest_dir=raw_dir):
    """Download one OSF file into dest_dir and check it against the OSF hash."""
    dest = dest_dir / row["name"]
    if dest.exists():
        if sha256sum(dest) != row["sha256"]:
            raise ValueError(f"{dest.name}: local copy differs from OSF; not overwriting")
        return "already present, hash verified"
    resp = session.get(row["download_url"], timeout=60)
    resp.raise_for_status()
    tmp = dest.with_name(dest.name + ".part")
    tmp.write_bytes(resp.content)
    if sha256sum(tmp) != row["sha256"]:
        tmp.unlink()
        raise ValueError(f"{dest.name}: downloaded file does not match OSF hash")
    tmp.rename(dest)
    return "downloaded, hash verified"


files["status"] = files.apply(download_verified, axis=1)
files[["node", "name", "status"]]

,node,name,status
0,5mnuq,PhD_MF_Age-neutrality_Archived.sav,"already present, hash verified"
1,hc9ur,OSF_LPFSBF20_PID5BF+M_Community.sav,"already present, hash verified"


## File structure

Only SPSS metadata is read here (`metadataonly=True`), so no responses are loaded.

In [8]:
structure = []
for _, row in files.iterrows():
    _, meta = pyreadstat.read_sav(raw_dir / row["name"], metadataonly=True)
    prefixes = pd.Series(meta.column_names).str.extract(r"^([A-Za-z]+)")[0]
    structure.append({
        "file": row["name"],
        "respondents": meta.number_rows,
        "variables": meta.number_columns,
        "variable groups": ", ".join(f"{k} ({v})" for k, v in prefixes.value_counts().items()),
    })
pd.DataFrame(structure)

,file,respondents,variables,variable groups
0,PhD_MF_Age-neutrality_Archived.sav,398,64,"PID (36), LPFSBF (12), CRS (12), Missings (2),..."
1,OSF_LPFSBF20_PID5BF+M_Community.sav,754,93,"PID (60), LPFS (18), CRS (12), Geslacht (1), L..."


{{< include closing.qmd >}} 